In [ ]:
from pathlib import Path
import sqlite3
import numpy as np
import pandas as pd

root = Path.cwd()
while not (root / "data" / "db").exists() and root != root.parent:
    root = root.parent

con = sqlite3.connect(root / "data" / "db" / "tennis.db")

perf = pd.read_sql("""
    SELECT e.match_id, e.tourney_start, e.player_id,
           e.result - e.expected AS r,
           m.surface
    FROM elo_long e
    JOIN matches m ON m.match_id = e.match_id
""", con)
players = pd.read_sql("SELECT player_id, name FROM players", con)

print(perf.shape)
print(perf["surface"].value_counts(dropna=False).to_string())

In [ ]:
def form(as_of, surface, window_days=1095, min_matches=10, max_gap_days=182, top=20):
    start = (pd.Timestamp(as_of) - pd.Timedelta(days=window_days)).strftime("%Y-%m-%d")
    d = perf[(perf["surface"] == surface)
             & (perf["tourney_start"] <= as_of)
             & (perf["tourney_start"] >= start)].copy()

    d = d.sort_values("match_id", ascending=False)
    d["i"] = d.groupby("player_id").cumcount()
    d["w"] = 0.5 ** (d["i"] / 10)
    d["w2"] = d["w"] ** 2
    d["wr"] = d["w"] * d["r"]

    g = d.groupby("player_id").agg(matches=("r", "size"), sw=("w", "sum"), sw2=("w2", "sum"),
                                   swr=("wr", "sum"), last_start=("tourney_start", "max"))
    g["form"] = g["swr"] / g["sw"]
    g["n_eff"] = g["sw"] ** 2 / g["sw2"]

    d = d.join(g["form"], on="player_id")
    d["res2"] = (d["w"] * (d["r"] - d["form"])) ** 2
    g["se"] = np.sqrt(d.groupby("player_id")["res2"].sum()) / g["sw"]

    cutoff = (pd.Timestamp(as_of) - pd.Timedelta(days=max_gap_days)).strftime("%Y-%m-%d")
    g = g[(g["matches"] >= min_matches) & (g["last_start"] >= cutoff)]
    g = g.sort_values("form", ascending=False).head(top)

    g = g.join(players.set_index("player_id")["name"])
    g["form_pct"] = (100 * g["form"]).round(1)
    g["plus_minus"] = (196 * g["se"]).round(1)
    g["n_eff"] = g["n_eff"].round(1)
    g.insert(0, "rank", range(1, len(g) + 1))
    return g[["rank", "name", "matches", "n_eff", "form_pct", "plus_minus"]]

In [ ]:
def form_table(as_of, surface, **kw):
    t = form(as_of, surface, top=1000, **kw)
    t["lower"] = (t["form_pct"] - t["plus_minus"]).round(1)
    t["upper"] = (t["form_pct"] + t["plus_minus"]).round(1)
    t["evidence"] = np.where(t["lower"] > 0, "above expectation",
                    np.where(t["upper"] < 0, "below expectation", "not enough evidence"))
    return t

full = form_table("2024-12-22", "Clay")
print(full.head(20)[["rank", "name", "matches", "form_pct", "lower", "upper", "evidence"]].to_string(index=False))
print(full["evidence"].value_counts().to_string())

In [ ]:
perf2 = pd.read_sql("""
    SELECT e.match_id, e.tourney_start, e.player_id, e.opponent_id,
           e.result - e.expected AS r, m.surface
    FROM elo_long e
    JOIN matches m ON m.match_id = e.match_id
""", con).sort_values(["player_id", "surface", "match_id"]).reset_index(drop=True)


def prior_form(df, half_life):
    lam = 0.5 ** (1 / half_life)
    pid = df["player_id"].to_numpy()
    sf = df["surface"].to_numpy()
    r = df["r"].to_numpy()
    num = np.zeros(len(df))
    den = np.zeros(len(df))
    cnt = np.zeros(len(df), dtype=int)
    n_, d_, c_ = 0.0, 0.0, 0
    for k in range(len(df)):
        if k == 0 or pid[k] != pid[k - 1] or sf[k] != sf[k - 1]:
            n_, d_, c_ = 0.0, 0.0, 0
        num[k], den[k], cnt[k] = n_, d_, c_
        n_ = r[k] + lam * n_
        d_ = 1.0 + lam * d_
        c_ += 1
    out = df[["match_id", "player_id", "opponent_id", "tourney_start", "r"]].copy()
    out["prior_n"] = cnt
    out["form_before"] = np.where(den > 0, num / np.where(den > 0, den, 1), np.nan)
    return out

print(perf2.shape)

In [ ]:
def slope_test(half_life, min_prev=5, start="2005-01-01", end="2019-01-01"):
    f = prior_form(perf2, half_life)
    f = f[(f["tourney_start"] >= start) & (f["tourney_start"] < end) & (f["prior_n"] >= min_prev)]
    opp = f[["match_id", "player_id", "form_before"]].rename(
        columns={"player_id": "opponent_id", "form_before": "opp_form"})
    mm = f.merge(opp, on=["match_id", "opponent_id"])
    mm = mm[mm["player_id"] < mm["opponent_id"]]

    d = (mm["form_before"] - mm["opp_form"]).to_numpy()
    y = mm["r"].to_numpy()
    X = np.column_stack([np.ones(len(d)), d])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta
    se = np.sqrt(resid.var(ddof=2) / ((d - d.mean()) ** 2).sum())
    return {"half_life": half_life, "matches": len(d),
            "slope": round(float(beta[1]), 3),
            "ci_low": round(float(beta[1] - 1.96 * se), 3),
            "ci_high": round(float(beta[1] + 1.96 * se), 3),
            "pts_per_10pts_form": round(float(10 * beta[1]), 2)}

print(pd.DataFrame([slope_test(h) for h in [5, 10, 20, 40]]).to_string(index=False))